# Full-sky HILC $y$ vs true Compton-$y$ power spectra

Compare the **harmonic ILC** Compton-$y$ map (pyILC HILC on the six homog HFI totals, **no Galactic/PS mask**) with the FLAMINGO **truth** Compton-$y$ map.

Observed maps: $d_\nu=B_\nu s+n$ (signal beamed, noise unbeamed). pyILC (McCarthy & Hill 2024) **beam-deconvolves** each channel and reconvolves to a common $10'$ Gaussian: $a_{\ell m}\to(B_{10'}/B_\nu)\,a_{\ell m}$. The $y$-map is therefore at $10'$. Spectra below **deconvolve that $10'$ beam** ($C_\ell^{yy}/B_{10'}^2$, $C_\ell^{yt}/B_{10'}$) so they sit on the same (unbeamed) footing as truth $y$. Noise follows the same operation: after ILC it is $(B_{10'}/B_\nu)n$, then $1/B_{10'}$ leaves $n/B_\nu$.

Full-sky `anafast`, mean-subtracted. **No** pixel-window deconvolution.

| | |
|---|---|
| ILC $y$ | `…/ilc/hilc_output_homog/flamingo_needletILCmap_component_tSZ_hilc_y_homog_fullsky.fits` |
| Truth $y$ | `…/components/tsz/compton_y_nside4096.fits` (ud_grade → 2048) |
| $\ell_\mathrm{max}$ | 4096 (pyILC `ELLMAX`) |
| Sky cut | none |


In [ ]:
from pathlib import Path

import healpy as hp
import matplotlib.pyplot as plt
import numpy as np

from flamingo_mock.powerspectra import bin_cl, compute_cl, dl_from_cl

NSIDE = 2048
LMAX = 4096
FWHM_ARCMIN = 10.0
BL_FLOOR = 1e-3
DELTA_ELL = 21

YMAP = Path(
    "/rds/rds-lxu/flamingo/integrated_maps_synthetic/ilc/hilc_output_homog"
    "/flamingo_needletILCmap_component_tSZ_hilc_y_homog_fullsky.fits"
)
TRUTH = Path(
    "/rds/rds-lxu/flamingo/integrated_maps_synthetic/components/tsz"
    "/compton_y_nside4096.fits"
)
_repo = Path("/scratch/scratch-lxu/flamingo_mock_analysis")
FIG_DIR = _repo / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("ILC y :", YMAP, "exists" if YMAP.is_file() else "MISSING")
print("truth :", TRUTH, "exists" if TRUTH.is_file() else "MISSING")


In [ ]:
def gaussian_bl(fwhm_arcmin, lmax):
    fwhm_rad = float(fwhm_arcmin) * np.pi / (180.0 * 60.0)
    sigma = fwhm_rad / np.sqrt(8.0 * np.log(2.0))
    ell = np.arange(lmax + 1, dtype=np.float64)
    return np.exp(-0.5 * ell * (ell + 1.0) * sigma * sigma)


if not YMAP.is_file():
    raise FileNotFoundError(
        f"HILC y-map not written yet:\n  {YMAP}\n"
        "Wait for logs/hilc_homog.log to print finished / y_map=."
    )

y_ilc = np.asarray(hp.read_map(str(YMAP), field=0, dtype=np.float64))
y_true = np.asarray(hp.read_map(str(TRUTH), field=0, dtype=np.float64))
if hp.get_nside(y_ilc) != NSIDE:
    y_ilc = hp.ud_grade(y_ilc, NSIDE)
if hp.get_nside(y_true) != NSIDE:
    y_true = hp.ud_grade(y_true, NSIDE)

print(
    f"nside={hp.get_nside(y_ilc)}  ILC rms={y_ilc.std():.4g}  "
    f"truth rms={y_true.std():.4g}"
)


In [ ]:
# Full-sky C_ell; leave pixel window in (as-observed at nside=2048).
cl_yy = compute_cl(y_ilc, lmax=LMAX, deconv_pixel_window=False)
cl_tt = compute_cl(y_true, lmax=LMAX, deconv_pixel_window=False)
cl_yt = compute_cl(y_ilc, y_true, lmax=LMAX, deconv_pixel_window=False)

ell = np.arange(LMAX + 1, dtype=np.float64)
bl = gaussian_bl(FWHM_ARCMIN, LMAX)
good = bl >= BL_FLOOR

cl_yy_dec = np.full_like(cl_yy, np.nan)
cl_yy_dec[good] = cl_yy[good] / bl[good] ** 2
cl_yt_dec = np.full_like(cl_yt, np.nan)
cl_yt_dec[good] = cl_yt[good] / bl[good]  # one B: ILC beamed, truth not
cl_tt_beamed = cl_tt * bl**2

ell_b, yy_b = bin_cl(cl_yy, delta_ell=DELTA_ELL)
_, tt_b = bin_cl(cl_tt, delta_ell=DELTA_ELL)
_, yt_b = bin_cl(cl_yt, delta_ell=DELTA_ELL)
_, yy_dec_b = bin_cl(cl_yy_dec, delta_ell=DELTA_ELL)
_, yt_dec_b = bin_cl(cl_yt_dec, delta_ell=DELTA_ELL)
_, tt_beam_b = bin_cl(cl_tt_beamed, delta_ell=DELTA_ELL)

with np.errstate(divide="ignore", invalid="ignore"):
    transfer_b = yt_dec_b / tt_b
    rho_b = yt_dec_b / np.sqrt(np.abs(yy_dec_b * tt_b))

lo, hi = 50, 500
band = (ell_b >= lo) & (ell_b <= hi)
print(f"median transfer C_yt/(B C_tt) in {lo}-{hi}: {np.nanmedian(transfer_b[band]):.3f}")
print(f"median rho_ell (deconv) in {lo}-{hi}: {np.nanmedian(rho_b[band]):.3f}")


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8.2, 8.2), sharex=True)

ax = axes[0]
ax.loglog(ell_b, dl_from_cl(ell_b, tt_b), color="k", lw=2, label=r"truth $y$ (unsmoothed)")
ax.loglog(
    ell_b, dl_from_cl(ell_b, tt_beam_b), color="0.55", lw=1.6,
    label=fr"truth $\times B_\ell^2$ ({FWHM_ARCMIN:.0f}')",
)
ax.loglog(ell_b, dl_from_cl(ell_b, yy_b), color="C0", lw=1.8, label=r"HILC $y$ (raw)")
ax.loglog(
    ell_b, dl_from_cl(ell_b, yy_dec_b), color="C0", lw=1.4, ls="--",
    label=r"HILC $y$ / $B_\ell^2$",
)
ax.loglog(ell_b, np.abs(dl_from_cl(ell_b, yt_b)), color="C1", lw=1.4, label=r"$|C_\ell^{yt}|$ (raw)")
ax.set_ylabel(r"$D_\ell = \ell(\ell+1)C_\ell/2\pi$")
ax.set_title("Full-sky HILC $y$ vs FLAMINGO truth Compton-$y$")
ax.legend(frameon=False, fontsize=9)
ax.set_xlim(2, LMAX)

ax = axes[1]
ax.axhline(1.0, color="k", lw=0.8)
ax.semilogx(ell_b, transfer_b, color="C1", lw=1.8, label=r"$C_\ell^{yt}/(B_\ell\,C_\ell^{tt})$")
ax.semilogx(ell_b, rho_b, color="C0", lw=1.5, ls="--", label=r"$\rho_\ell$ (beam-deconv)")
ax.set_ylim(0.0, 1.6)
ax.set_xlabel(r"$\ell$")
ax.set_ylabel("transfer / correlation")
ax.legend(frameon=False, fontsize=9)

fig.tight_layout()
out = FIG_DIR / "hilc_homog_fullsky_y_vs_truth_ps.png"
fig.savefig(out, dpi=150)
print("wrote", out)
plt.show()


## Blind CIB + noise residuals (Planck 2015 XXII)

HILC is **tSZ-preserving** with `N_deproj=0`: it does **not** use a CIB SED in the filter. Residual CIB is estimated **blindly**, as in Planck 2015 XXII §6: assume a CIB model (not the FLAMINGO CIB map) and apply the **same $y$-map weights**.

**CIB model (P15).** Clustered + Poisson CIB from Planck XXX (2014) Table 15 auto $C_\ell$ in $\mathrm{Jy}^2\,\mathrm{sr}^{-1}$ (halo model + Béthermin et al. 2012 shot noise; that is what P15 injects into simulations). Cross-spectra use Planck XXX Table 11 frequency decoherence $r_{\nu\nu'}$. Convert $\mathrm{Jy}\to K_\mathrm{CMB}$ with $dB_\nu/dT$ (same as the mock). Then

$$C_\ell^{y,\mathrm{CIB}}=\sum_{ij} w_i(\ell)\,w_j(\ell)\,C_\ell^{ij,K},$$

with $w_i$ the raw HILC tSZ weights (zero if that channel is dropped). Bolliet et al. (2018) float $A_{\mathrm{CIB}}$ in front of the P15 residual template; here $A_{\mathrm{CIB}}=1$.

**Noise.** Homogeneous white $N_\ell$ from the Planck $w^{-1/2}$ table, uncorrelated between channels. After 10' beam deconvolution: $C_\ell^{N,y}/B_\ell^2=\sum_\nu (w_\nu/B_\nu)^2 N_\ell^\nu$.

All curves below are **beam-deconvolved** by the 10' ILC beam.


In [ ]:
from flamingo_mock.config import BEAM_FWHM_ARCMIN
from flamingo_mock.spectral import dB_dT_Jy_per_sr_per_K

FREQS = (100, 143, 353, 217, 545, 857)  # YAML / weight order
FWHM_CH = [BEAM_FWHM_ARCMIN[int(f)] for f in FREQS]
NELL_UK2 = {100: 5.07e-4, 143: 9.21e-5, 353: 2.00e-3, 217: 1.85e-4, 545: 5.51e-2, 857: 30.9}
BINSIZE, BEAM_CRIT = 50, 1.0e-3
WDIR = Path("/rds/rds-lxu/flamingo/integrated_maps_synthetic/ilc/hilc_output_homog")
idx = {f: i for i, f in enumerate(FREQS)}

ellbins = np.arange(0, LMAX + 1, BINSIZE)
n_scales = len(ellbins) - 1
filts = np.zeros((n_scales, LMAX + 1))
for i in range(n_scales - 1):
    filts[i, ellbins[i] : ellbins[i + 1]] = 1.0
filts[-1, ellbins[-1] :] = 1.0
ells = np.arange(LMAX + 1, dtype=np.float64)
inp_beams = [hp.gauss_beam(np.deg2rad(f / 60.0), lmax=LMAX) for f in FWHM_CH]
ell_B = np.array([int(np.argmin(np.abs(b - BEAM_CRIT))) for b in inp_beams])
ell_F = np.zeros(n_scales)
for i in range(n_scales - 1):
    peak = int(np.argmax(filts[i]))
    ell_F[i] = min(LMAX, peak + int(np.argmin(np.abs(filts[i][peak:] - BEAM_CRIT))))
ell_F[-1] = ell_F[-2]

# Raw HILC weights vs ell (channel dropped => 0).
w_ell = np.zeros((len(FREQS), LMAX + 1))
for j in range(n_scales):
    use = [ell_F[j] <= ell_B[a] for a in range(len(FREQS))]
    wraw = np.atleast_1d(
        np.loadtxt(WDIR / f"flamingo_weightvector_scale{j}_component_tSZ.txt")
    ).ravel()
    sl = filts[j] > 0
    count = 0
    for a, ok in enumerate(use):
        if not ok:
            continue
        w_ell[a, sl] = wraw[count]
        count += 1

# Planck XXX (2014) Table 15 auto C_ell [Jy^2/sr]; Table 6+7 shot (dust+radio).
tab_ell = np.array([187.0, 320.0, 502.0, 684.0, 890.0, 1158.0, 1505.0, 1956.0, 2649.0])
tab = {
    857: np.array([2.87e5, 1.34e5, 7.20e4, 4.38e4, 3.23e4, 2.40e4, 1.83e4, 1.46e4, 1.16e4]),
    545: np.array([6.63e4, 3.34e4, 1.91e4, 1.25e4, 9.17e3, 6.83e3, 5.34e3, 4.24e3, 3.42e3]),
    353: np.array([7.88e3, 4.35e3, 2.60e3, 1.74e3, 1.29e3, 9.35e2, 7.45e2, 6.08e2, np.nan]),
    217: np.array([4.17e2, 2.62e2, 1.75e2, 1.17e2, 8.82e1, 6.42e1, 3.34e1, 4.74e1, np.nan]),
    143: np.array([3.64e1, 3.23e1, 2.81e1, 2.27e1, 1.84e1, 1.58e1, 1.25e1, np.nan, np.nan]),
}
shot = {100: 8.62, 143: 7.25, 217: 19.12, 353: 228.28, 545: 1456.86, 857: 5632.28}


def interp_auto(nu, ell):
    y = tab[nu]
    m = np.isfinite(y)
    le, ye = tab_ell[m], y[m]
    out = np.empty_like(ell, dtype=float)
    for i, L in enumerate(ell):
        if L < 2:
            out[i] = 0.0
        elif L < le[0]:
            out[i] = ye[0] * (L / le[0]) ** -1.2
        elif L > le[-1]:
            clust = max(ye[-1] - shot[nu], 0.0) * (L / le[-1]) ** -1.2
            out[i] = shot[nu] + clust
        else:
            out[i] = np.exp(np.interp(np.log(L), np.log(le), np.log(ye)))
    return out


R = np.eye(len(FREQS))
for (a, b), rij in {
    (143, 217): 0.78, (143, 353): 0.54, (143, 545): 0.51, (143, 857): 0.45,
    (217, 353): 0.91, (217, 545): 0.90, (217, 857): 0.85,
    (353, 545): 0.983, (353, 857): 0.911, (545, 857): 0.949,
    (100, 143): 0.70, (100, 217): 0.50, (100, 353): 0.40, (100, 545): 0.35, (100, 857): 0.30,
}.items():
    i, j = idx[a], idx[b]
    R[i, j] = R[j, i] = rij

k_per_jy = np.array([1.0 / dB_dT_Jy_per_sr_per_K(f) for f in FREQS])
cl_jy = np.zeros((len(FREQS), LMAX + 1))
for nu in (143, 217, 353, 545, 857):
    cl_jy[idx[nu]] = interp_auto(nu, ells)
clust143 = np.maximum(cl_jy[idx[143]] - shot[143], 0.0)
cl_jy[idx[100]] = shot[100] + 0.2 * clust143
cl_jy[:, :2] = 0.0

cl_cib_dec = np.full(LMAX + 1, np.nan)
cl_n_dec = np.full(LMAX + 1, np.nan)
for L in np.where(good)[0]:
    cj = cl_jy[:, L]
    Ck = np.outer(k_per_jy, k_per_jy) * (R * np.sqrt(np.outer(cj, cj)))
    w = w_ell[:, L]
    cl_cib_dec[L] = w @ Ck @ w
    cl_n_dec[L] = sum(
        (w[a] / max(inp_beams[a][L], 1e-30)) ** 2 * (NELL_UK2[f] * 1e-12)
        for a, f in enumerate(FREQS)
    )

_, cib_b = bin_cl(cl_cib_dec, delta_ell=DELTA_ELL)
_, n_b = bin_cl(cl_n_dec, delta_ell=DELTA_ELL)
dl_tot = dl_from_cl(ell_b, yy_dec_b)
dl_cib = dl_from_cl(ell_b, cib_b)
dl_n = dl_from_cl(ell_b, n_b)

print("ell   D_total      D_CIB        D_noise     CIB/total")
for L0 in (50, 100, 300, 500, 1000, 1500, 2000):
    i = int(np.nanargmin(np.abs(ell_b - L0)))
    t, c, n = dl_tot[i], dl_cib[i], dl_n[i]
    print(f"{ell_b[i]:5.0f}  {t:10.3e}  {c:10.3e}  {n:10.3e}  {c / t:7.3f}")

fig, ax = plt.subplots(figsize=(8.2, 4.8))
ax.loglog(ell_b, dl_tot, "C0", lw=2.2, label=r"HILC total $y$ / $B_\ell^2$")
ax.loglog(ell_b, dl_cib, "C3", lw=2.0, label=r"CIB residual (P15 / Planck XXX $\times$ weights)")
ax.loglog(ell_b, dl_n, color="0.35", lw=2.0, ls="--", label=r"instrumental noise residual")
ax.set_xlim(10, LMAX)
ax.set_xlabel(r"$\ell$")
ax.set_ylabel(r"$D_\ell=\ell(\ell+1)C_\ell/2\pi$")
ax.set_title("Beam-deconvolved HILC $y$: total, Planck 2015 CIB, noise")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
out = FIG_DIR / "hilc_homog_fullsky_y_cib_noise_residual.png"
fig.savefig(out, dpi=150)
print("wrote", out)
plt.show()


## Yang et al. (2026) CIB vs map-inferred residual

Separate from P15: two more $y$-space CIB residuals with the **same** HILC weights.

1. **Yang 2026 model.** Auto $C_\ell$ of the FLAMINGO three-parameter CIB maps (Yang et al. 2026), cross-spectra from their Table 3 $r_{\nu\nu'}$ (217–857 GHz; 100/143 GHz use the 217 GHz row as a proxy). Then $C_\ell^{y}=\sum_{ij}w_i w_j C_\ell^{ij,K}$. This is the P15 procedure with Yang's $C_\ell$ and decoherence instead of Planck XXX.
2. **Map-inferred.** Apply the weights in harmonic space to the six `CIB_deltaT_*` maps (the actual leakage in this mock). Unbeamed CIB in $K_\mathrm{CMB}$, so $a_{\ell m}^{y,\mathrm{CIB}}=\sum_\nu w_\nu(\ell)\,B_\ell^{10'}\,a_{\ell m}^{\mathrm{CIB},\nu}$.

Compare amplitudes to P15 at $\ell=50$–$500$.

**Budget.** Overlay $y_{\mathrm{th}}+\mathrm{CIB}_{\mathrm{map}}+N$ on the total (map CIB is the actual leakage).


In [ ]:
CIB_DIR = Path("/rds/rds-lxu/flamingo/integrated_maps_synthetic/components/cib")
# Yang et al. 2026 Table 3 (this lightcone, 150 < ell < 1000).
R_YANG = np.eye(len(FREQS))
for (a, b), rij in {
    (217, 353): 0.993, (217, 545): 0.956, (217, 857): 0.841,
    (353, 545): 0.983, (353, 857): 0.895, (545, 857): 0.959,
    # 100/143 not in Table 3: treat as 217-like (Yang SED-scaled, highly correlated).
    (100, 143): 0.95, (100, 217): 0.95, (143, 217): 0.95,
    (100, 353): 0.993, (143, 353): 0.993,
    (100, 545): 0.956, (143, 545): 0.956,
    (100, 857): 0.841, (143, 857): 0.841,
}.items():
    i, j = idx[a], idx[b]
    R_YANG[i, j] = R_YANG[j, i] = rij

alms = []
cl_cib_maps = np.zeros((len(FREQS), LMAX + 1))
y_alm = None
for a, f in enumerate(FREQS):
    p = CIB_DIR / f"CIB_deltaT_{f}GHz_nside4096.fits"
    m = np.asarray(hp.read_map(str(p), dtype=np.float64))
    m = hp.ud_grade(m, NSIDE) * 1e-6  # uK -> K, same nside as ILC
    m -= np.mean(m)
    alm = hp.map2alm(m, lmax=LMAX, iter=0)
    alms.append(alm)
    cl_cib_maps[a] = hp.alm2cl(alm)
    contrib = hp.almxfl(alm, w_ell[a] * bl)
    y_alm = contrib if y_alm is None else y_alm + contrib
    print(f"CIB {f} GHz  rms={m.std():.3e} K")
    del m

cl_map = hp.alm2cl(y_alm)
cl_map_dec = np.full(LMAX + 1, np.nan)
cl_map_dec[good] = cl_map[good] / bl[good] ** 2
del y_alm, alms

cl_yang_dec = np.full(LMAX + 1, np.nan)
for L in np.where(good)[0]:
    cj = np.maximum(cl_cib_maps[:, L], 0.0)
    Ck = R_YANG * np.sqrt(np.outer(cj, cj))
    cl_yang_dec[L] = w_ell[:, L] @ Ck @ w_ell[:, L]

_, yang_b = bin_cl(cl_yang_dec, delta_ell=DELTA_ELL)
_, map_b = bin_cl(cl_map_dec, delta_ell=DELTA_ELL)
dl_yang = dl_from_cl(ell_b, yang_b)
dl_map = dl_from_cl(ell_b, map_b)

from pyilc.fg import get_mix

SED_YML = "/scratch/scratch-lxu/agent_dev/auto_research_agent/pyilc/input/fg_SEDs_default_params.yml"
a_tsz = 1e-6 * np.array(
    [get_mix([float(f)], "tSZ", param_dict_file=SED_YML)[0] for f in FREQS]
)
g_tsz = a_tsz @ w_ell
cl_th_dec = np.full(LMAX + 1, np.nan)
cl_th_dec[good] = g_tsz[good] ** 2 * cl_tt[good]
_, th_b = bin_cl(cl_th_dec, delta_ell=DELTA_ELL)
dl_th = dl_from_cl(ell_b, th_b)
print(
    "g=sum w a_tSZ: "
    f"ell50={g_tsz[50]:.4f}  ell300={g_tsz[300]:.4f}  ell1500={g_tsz[1500]:.4f}"
)

band = (ell_b >= 50) & (ell_b <= 500) & np.isfinite(dl_cib) & np.isfinite(dl_yang) & np.isfinite(dl_map)
r_yp = np.nanmedian(dl_yang[band] / dl_cib[band])
r_mp = np.nanmedian(dl_map[band] / dl_cib[band])
r_my = np.nanmedian(dl_map[band] / dl_yang[band])
print(f"ell=50-500  Yang/P15={r_yp:.2f}  maps/P15={r_mp:.2f}  maps/Yang={r_my:.2f}")
print("ell   D_P15        D_Yang       D_maps      Yang/P15  maps/P15")
for L0 in (50, 100, 300, 500, 1000, 1500):
    i = int(np.nanargmin(np.abs(ell_b - L0)))
    print(
        f"{ell_b[i]:5.0f}  {dl_cib[i]:10.3e}  {dl_yang[i]:10.3e}  {dl_map[i]:10.3e}"
        f"  {dl_yang[i]/dl_cib[i]:7.2f}  {dl_map[i]/dl_cib[i]:7.2f}"
    )

dl_sum_map = dl_th + dl_map + dl_n

fig, ax = plt.subplots(figsize=(8.2, 4.8))
ax.loglog(ell_b, dl_tot, "C0", lw=2.2, label=r"HILC total $y$ / $B_\ell^2$")
ax.loglog(
    ell_b, dl_th, color="k", lw=1.8, ls="-.",
    label=r"theoretical $y$ ($\sum_\nu w_\nu a_\nu^{\mathrm{tSZ}}\times$ truth)",
)
ax.loglog(ell_b, dl_cib, "C3", lw=2.0, label=r"CIB P15 (Planck XXX $\times$ weights)")
ax.loglog(ell_b, dl_yang, "C2", lw=1.8, ls="--", label=r"CIB Yang 2026 ($C_\ell$ + Table 3 $\times$ weights)")
ax.loglog(ell_b, dl_map, "C1", lw=1.8, label=r"CIB from maps (weights $\times$ FLAMINGO CIB)")
ax.loglog(ell_b, dl_n, color="0.35", lw=1.6, ls=":", label=r"instrumental noise residual")
ax.loglog(
    ell_b, dl_sum_map, color="C4", lw=1.5, ls="--",
    label=r"$y_{\mathrm{th}}+\mathrm{CIB}_{\mathrm{map}}+N$",
)
ax.set_xlim(10, LMAX)
ax.set_xlabel(r"$\ell$")
ax.set_ylabel(r"$D_\ell=\ell(\ell+1)C_\ell/2\pi$")
ax.set_title("Beam-deconvolved HILC $y$: P15 vs Yang 2026 vs map CIB")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
out = FIG_DIR / "hilc_homog_fullsky_y_cib_noise_residual.png"
fig.savefig(out, dpi=150)
print("wrote", out)
plt.show()
